In [3]:
import json
import os
import time
import openai
from tqdm import tqdm 

with open("tools_metadata_with_gen.json", "r", encoding="utf-8") as f:
    data = json.load(f)

def prompt_scadsai_llm(tool_context, model="openai/gpt-oss-120b"):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    system_msg = "You are a precise, factual technical writer. Use only provided metadata."

    user_msg = (
        "You're a technical writer. Be precise, fact-based, and objective. "
        "You may infer reasonable purpose and usage from the text, but do not invent unrelated details. "
        "Use only the fields 'name', 'description', 'repo_description', and 'repo_long_description' "
        "from the JSON below to summarize the tool in 3–5 sentences:\n\n"
        f"{json.dumps(tool_context, ensure_ascii=False, indent=2)}"
    )
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="https://llm.scads.ai/v1",
                           api_key=os.environ.get('SCADSAI_API_KEY')
    )
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},]
    )
    
    # extract answer
    return response.choices[0].message.content

seen = {}
for entry in data:
    for tool in entry.get("tools", []):
        guid = tool.get("guid")
        if guid and tool.get("detailed_description_generated"):
            seen[guid] = tool["detailed_description_generated"]


processed = 0
checkpoint_every = 50
total_tools = sum(len(e.get("tools", [])) for e in data)
print(f"Bereits vorhanden (Cache): {len(seen)} Tool-Beschreibungen. Gesamt-Tools: {total_tools}")

tools_all = [(entry, tool) for entry in data for tool in entry.get("tools", [])]
for entry, tool in tqdm(tools_all, desc="Generiere Beschreibungen", unit="tool"):
    for tool in entry.get("tools", []):
        guid = tool.get("guid")

        # Bereits erledigt? -> überspringen
        if tool.get("detailed_description_generated"):
            processed += 1
            continue
        if guid in seen:
            tool["detailed_description_generated"] = seen[guid]
            processed += 1
            continue

        # Kontext bauen
        ctx = {
            "repository_owner": entry.get("owner"),
            "repository_name": entry.get("name"),
            "changeset_revision": entry.get("changeset_revision"),
            "tool_shed_url": entry.get("tool_shed_url"),
            "tool_id": tool.get("tool_id"),
            "name": tool.get("name"),
            "version": tool.get("version"),
            "description": tool.get("description"),
            "repo_description": tool.get("repo_description"),
            "repo_long_description": tool.get("repo_long_description"),
            "guid": guid,
        }

        try:
            desc = prompt_scadsai_llm(ctx)
            tool["detailed_description_generated"] = desc
            if guid:
                seen[guid] = desc

        except Exception as e:
            print(f"Fehler bei {tool.get('name') or tool.get('tool_id')}: {e}")
            tool["detailed_description_generated"] = f"Fehler: {e}"

        processed += 1

       
        time.sleep(1)

      
        if processed % checkpoint_every == 0:
            with open("tools_metadata_with_gen.json", "w", encoding="utf-8") as out:
                json.dump(data, out, ensure_ascii=False, indent=2)



with open("tool_metadata_gen.json", "w", encoding="utf-8") as out:
    json.dump(data, out, ensure_ascii=False, indent=2)
print(f"Done. Wrote tool_metadata_gen.json")

Bereits vorhanden (Cache): 3786 Tool-Beschreibungen. Gesamt-Tools: 17943


Generiere Beschreibungen: 100%|██████████| 17943/17943 [00:00<00:00, 384136.04tool/s]


Done. Wrote tool_metadata_gen.json


In [4]:
import json
import os
import time
from tqdm import tqdm
import openai

# === Konfiguration ===
INPUT = "tools_metadata_with_gen.json"
OUTPUT = "tools_metadata_with_gen.json"
N_LAST = 19  # wie viele Tools am Ende neu generiert werden sollen

# === LLM-Verbindung ===
def prompt_scadsai_llm(tool_context, model="openai/gpt-oss-120b"):
    """Sendet Tool-Kontext an ScaDS.AI-LLM und gibt Textbeschreibung zurück."""
    system_msg = "You are a precise, factual technical writer. Use only provided metadata."
    user_msg = (
        "You're a technical writer. Be precise, fact-based, and objective. "
        "You may infer reasonable purpose and usage from the text, but do not invent unrelated details. "
        "Use only the fields 'name', 'description', 'repo_description', and 'repo_long_description' "
        "from the JSON below to summarize the tool in 3–5 sentences:\n\n"
        f"{json.dumps(tool_context, ensure_ascii=False, indent=2)}"
    )

    client = openai.OpenAI(
        base_url="https://llm.scads.ai/v1",
        api_key=os.environ.get("SCADSAI_API_KEY"),
    )
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
    )
    return response.choices[0].message.content


# === Datei laden ===
with open(INPUT, "r", encoding="utf-8") as f:
    data = json.load(f)

# Alle Tools linear auflisten
flat_tools = []
for ri, repo in enumerate(data):
    for ti, tool in enumerate(repo.get("tools", [])):
        flat_tools.append((ri, ti, tool))

print(f"Gesamt: {len(flat_tools)} Tools. Regeneriere die letzten {N_LAST} ...")

# Die letzten N Tools nehmen
tail = flat_tools[-N_LAST:]

# === Neu generieren ===
for ri, ti, tool in tqdm(tail, desc="Regeneriere letzte Tools", unit="tool"):
    r = data[ri]
    ctx = {
        "repository_owner": r.get("owner"),
        "repository_name": r.get("name"),
        "changeset_revision": r.get("changeset_revision"),
        "tool_shed_url": r.get("tool_shed_url"),
        "tool_id": tool.get("tool_id"),
        "name": tool.get("name"),
        "version": tool.get("version"),
        "description": tool.get("description"),
        "repo_description": tool.get("repo_description"),
        "repo_long_description": tool.get("repo_long_description"),
        "guid": tool.get("guid"),
    }

    try:
        desc = prompt_scadsai_llm(ctx)
        tool["detailed_description_generated"] = desc
        print(f"✅ {tool.get('name')} aktualisiert")
    except Exception as e:
        tool["detailed_description_generated"] = f"Fehler: {e}"
        print(f"⚠️ Fehler bei {tool.get('name')}: {e}")

    time.sleep(1)  # kleine Pause gegen Rate-Limits

# === Speichern ===
with open(OUTPUT, "w", encoding="utf-8") as out:
    json.dump(data, out, ensure_ascii=False, indent=2)

print(f"\nFertig. Letzte {N_LAST} Tools neu beschrieben und in {OUTPUT} gespeichert.")


Gesamt: 17943 Tools. Regeneriere die letzten 19 ...


Regeneriere letzte Tools:   0%|          | 0/19 [00:00<?, ?tool/s]

✅ YAHS aktualisiert


Regeneriere letzte Tools:   5%|▌         | 1/19 [00:06<01:48,  6.03s/tool]

✅ YAHS aktualisiert


...

✅ zerone aktualisiert


Regeneriere letzte Tools:  95%|█████████▍| 18/19 [01:39<00:05,  5.51s/tool]

✅ Zoo Project OGC API Processes aktualisiert


Regeneriere letzte Tools: 100%|██████████| 19/19 [01:45<00:00,  5.55s/tool]



Fertig. Letzte 19 Tools neu beschrieben und in tools_metadata_with_gen.json gespeichert.


dedupe:

In [5]:
import json

INPUT = "000_d_tools_metadata_with_gen.json"
OUTPUT = "000_e_tools_metadata_deduped.json"

# Datei laden
with open(INPUT, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Vorher: {len(data)} Repositories")

# Wir definieren einen eindeutigen Schlüssel pro Repo:
# (owner, name, changeset_revision)
seen = set()
deduped = []

for repo in data:
    key = (
        repo.get("owner"),
        repo.get("name"),
        repo.get("changeset_revision"),
    )

    # Prüfen, ob der Schlüssel schon existiert
    if key not in seen:
        seen.add(key)
        deduped.append(repo)
    else:
        print(f"Duplikat entfernt: {key}")

# Ergebnis speichern
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(deduped, f, ensure_ascii=False, indent=2)

print(f"Nachher: {len(deduped)} Repositories")
print(f"Gespeichert in: {OUTPUT}")


Vorher: 14099 Repositories
Duplikat entfernt: ('imgteam', '2d_auto_threshold', '50fa6150e340')
Duplikat entfernt: ('imgteam', '2d_auto_threshold', '50fa6150e340')
Duplikat entfernt: ('imgteam', '2d_auto_threshold', '50fa6150e340')
Duplikat entfernt: ('imgteam', '2d_auto_threshold', '50fa6150e340')
Duplikat entfernt: ('imgteam', '2d_auto_threshold', '50fa6150e340')
Duplikat entfernt: ('imgteam', '2d_auto_threshold', '50fa6150e340')
Duplikat entfernt: ('imgteam', '2d_auto_threshold', '50fa6150e340')
Duplikat entfernt: ('imgteam', '2d_feature_extraction', '5bc8cdc17fd0')
Duplikat entfernt: ('imgteam', '2d_feature_extraction', '5bc8cdc17fd0')
Duplikat entfernt: ('imgteam', '2d_feature_extraction', '5bc8cdc17fd0')
Duplikat entfernt: ('imgteam', '2d_feature_extraction', '5bc8cdc17fd0')
Duplikat entfernt: ('imgteam', '2d_feature_extraction', '5bc8cdc17fd0')
Duplikat entfernt: ('imgteam', '2d_filter_segmentation_by_features', '9d47aabda459')
Duplikat entfernt: ('imgteam', '2d_filter_segmentati